In [1]:
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import RMSE, MAE, QuantileLoss

# Load processed dataset
data = pd.read_parquet("sales_features.parquet").reset_index()

data["sale_dollars"] = pd.to_numeric(data["sale_dollars"], errors="coerce")

# Convert date to datetime and sort
data["date"] = pd.to_datetime(data["date"])
data = data.sort_values(["store", "date"])

# Create time index (consecutive integers for each store's timeline)
data["time_idx"] = data.groupby("store").cumcount() + 1

# Define forecasting parameters
max_encoder_length = 30  # Use 90 days of history
max_prediction_length = 7  # Forecast 7 days ahead
training_cutoff = data["time_idx"].max() - max_prediction_length  # Split point]

categorical_columns = [
    # Static categoricals
    "store", "city", "county", "store_size", "zipcode", "holiday_name",
    # Time-varying categoricals
    "month", "quarter", "year", "day_of_week", "is_weekend", "is_holiday"
]

for col in categorical_columns:
    data[col] = data[col].astype(str)  # Convert to string type

# Handle infinite values
numeric_cols = data.select_dtypes(include=[np.number]).columns
data[numeric_cols] = data[numeric_cols].replace([np.inf, -np.inf], np.nan)
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].median())

data["sale_dollars"] = data["sale_dollars"].replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=["sale_dollars"])
print("NaNs after explicit drop:", data["sale_dollars"].isna().sum())

store_time_counts = data.groupby("store")["time_idx"].count()
print("Stores with insufficient timesteps:", store_time_counts[store_time_counts < max_encoder_length].count())

store_time_counts = data.groupby("store")["time_idx"].count()
valid_stores = store_time_counts[store_time_counts >= max_encoder_length + max_prediction_length].index

# Filter data
data = data[data["store"].isin(valid_stores)]

# For each store, create a complete date range and reindex
all_dates = pd.date_range(data["date"].min(), data["date"].max(), freq="D")
data = data.groupby("store").apply(
    lambda group: group.set_index("date").reindex(all_dates).ffill().reset_index()
).reset_index(drop=True)

# Recalculate time_idx after filling gaps
data["time_idx"] = data.groupby("store").cumcount() + 1

# Drop stores where all sale_dollars are NaN
invalid_stores = data.groupby("store")["sale_dollars"].apply(lambda x: x.isna().all())
invalid_stores = invalid_stores[invalid_stores].index
data = data[~data["store"].isin(invalid_stores)]

# After filtering stores, check remaining data
print(f"Stores remaining: {data['store'].nunique()}")
print(f"Min timesteps per store: {data.groupby('store')['time_idx'].count().min()}")

# Define dataset parameters
training = TimeSeriesDataSet(
    data[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="sale_dollars",
    group_ids=["store"],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=[
        "store", "city", "county", "store_size", 
        "zipcode", "holiday_name"
    ],
    static_reals=[
        "lon", "lat", "store_avg_sales",
        "city_avg_sales", "county_avg_sales",
        "store_avg_transactions"
    ],
    time_varying_known_categoricals=[
        "month", "quarter", "year", 
        "day_of_week", "is_weekend", "is_holiday"
    ],
    time_varying_known_reals=[
        "day_of_month", "week_of_year",
        "days_to_nearest_holiday"
    ],
    time_varying_unknown_categoricals=[],
    time_varying_unknown_reals=[
        "sale_dollars", "sale_bottles",
        "sale_liters", "transaction_count",
        "avg_price_per_bottle", "profit_margin",
        "avg_transaction_value",
        "sale_dollars_rolling_mean_7D",
        "sale_dollars_rolling_mean_30D",
        "sale_dollars_momentum_7D",
        "sale_dollars_significant_decrease"
    ],
    target_normalizer=GroupNormalizer(
        groups=["store"], transformation="softplus"
    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

# Create validation and training dataloaders
validation = TimeSeriesDataSet.from_dataset(
    training, data, predict=True, stop_randomization=True
)
batch_size = 64
train_dataloader = training.to_dataloader(
    train=True, batch_size=batch_size, num_workers=4
)
val_dataloader = validation.to_dataloader(
    train=False, batch_size=batch_size * 10, num_workers=4
)

# Configure TFT model
pl.seed_everything(42)
early_stop_callback = EarlyStopping(
    monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min"
)

trainer = pl.Trainer(
    max_epochs=50,
    gpus=1 if torch.cuda.is_available() else 0,
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback],
)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,
    output_size=7,
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=3,
    allow_missing_timesteps=True
)

# Train the model
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

# Evaluate performance on test data
best_model_path = trainer.checkpoint_callback.best_model_path
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)

actuals = torch.cat([y[0] for x, y in iter(val_dataloader)])
predictions = best_tft.predict(val_dataloader)
rmse = RMSE()(predictions, actuals)
mae = MAE()(predictions, actuals)
print(f"Test RMSE: {rmse:.2f}, Test MAE: {mae:.2f}")

# Visualize predictions vs actuals
raw_predictions, x = best_tft.predict(val_dataloader, mode="raw", return_x=True)
best_tft.plot_prediction(x, raw_predictions, idx=0, add_loss_to_title=True)

NaNs after explicit drop: 0
Stores with insufficient timesteps: 430
Stores remaining: 1904
Min timesteps per store: 264


AssertionError: Timeseries index should be of type integer